In [0]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.preprocessing import OrdinalEncoder
from scipy.stats import entropy

In [0]:
# source data file path
source_data_path = "/Volumes/workspace/default/source_data/source_data.csv"

# database to use
db_name = "default" 
spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")
spark.sql(f"USE {db_name}")

current_catalog = spark.catalog.currentCatalog()
print(f"Current Catalog: {current_catalog}")
current_db = spark.catalog.currentDatabase()
print(f"Current Database: {current_db}")



# Bronze
Requirements:
- all records and columns ingested

In [0]:
# raw data ingestion

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "") \
    .load(source_data_path)

df.write.format("delta").mode("overwrite").saveAsTable("bronze_data")

display(df)

# Silver
Requirements:
- contains only full records
- contains only valid records
- column names expanded where necessary

In [0]:
df = spark.table("bronze_data")

In [0]:
# drop columns that exceed the missing value percentage defined in the threshold

THRESHOLD = 90 

column_stats = []
columns_to_drop = []

df_count = df.count()

for column in df.columns:

    col_count = df.filter(df[column].isNull()).count()
    percentage = round((col_count / df_count) * 100, 3)
    drop = percentage >= THRESHOLD

    column_stats.append({
        "name": column,
        "number_of_missing_values": col_count,
        "percentage_of_missing_values": percentage,
        "drop": drop})
    
    if drop:
        columns_to_drop.append(column)

column_stats_df = pd.DataFrame(column_stats)

print("Summary of missing values by column:")
display(column_stats_df.sort_values(by="number_of_missing_values", ascending=False))

df = df.drop(*columns_to_drop)

print(f"Dropped columns:\n{',\n'.join(columns_to_drop)}\n")
print(f"Remaining columns:\n{',\n'.join(df.columns)}")


In [0]:
# drop rows containing missing values

temp_df_count = df_count
df = df.dropna()
df_count = df.count()

print(f"Number of dropped rows: {temp_df_count - df_count}")
print("Remaining rows:")
display(df)

In [0]:
# expand column name where necessary

rename_map = {
    "oph": "operating_hours",
    "pist_m": "piston_material",
    "issue_type": "combustion_issue_type",
    "bmep": "break_mean_effective_pressure",
    "ng_imp": "natural_gas_impurities_nmol",
    "past_dmg": "had_past_damages",
    "rpm_max": "maximum_rotations_per_minute",
    "full_load_issues": "had_full_load_issues",
    "number_up": "number_of_unplanned_events",
    "number_tc": "number_of_turbo_chargers",
    "op_set_1": "operational_setting_1",
    "op_set_2": "operational_setting_2",
    "op_set_3": "operational_setting_3",
    "high_breakdown_risk": "chance_of_breakdown" # rename to better align with the business description
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("silver_data")

#Gold 
Requirements:
- keeps columns that contain information
- keeps records that passed **input** and **business tests**

**Input tests**:
- records with mismatching types 
- records with invalid values
- records with missing values

**Business tests**:
- OPH <= 120000

In [0]:
df = spark.table("silver_data")

In [0]:
# drop columns that contains only just a single value 

columns_to_drop = []

for column in df.columns:
        
    if df.select(column).distinct().count() == 1:
        columns_to_drop.append(column)

df = df.drop(*columns_to_drop)

print(f"Dropped columns:\n{',\n'.join(columns_to_drop)}\n")
print(f"Remaining columns:\n{',\n'.join(df.columns)}")


In [0]:
# cast columns to their expected data types 
# and drop rows where casting was unsuccessful

type_map = {
    "operating_hours": "integer",            
    "piston_material": "integer",            
    "combustion_issue_type": "string",     
    "break_mean_effective_pressure": "double",          
    "natural_gas_impurities_nmol": "double",         
    "had_past_damages": "boolean",          
    "resting_analysis_results": "string",
    "maximum_rotations_per_minute": "double",
    "had_full_load_issues": "boolean",
    "number_of_unplanned_events": "integer",
    "number_of_turbo_chargers": "integer",
    "operational_setting_1": "double",
    "operational_setting_2": "double",
    "operational_setting_3": "double",
    "chance_of_breakdown": "string"
}

for column, cast_type in type_map.items():
    if column in df.columns:
        df = df.withColumn(column, df[column].try_cast(cast_type))

print("New schema:")
df.printSchema()

temp_df_count = df.count()
df = df.dropna()
df_count = df.count()

print(f"Number of dropped rows: {temp_df_count - df_count}")
print("Remaining rows:")
display(df)

In [0]:
# replace category names where necessary
# and drop records with invalid values

# map chance of breakdown values 
# to get aligned with the business description
category_map = {           
    "combustion_issue_type": {
        "1": "typical",
        "2": "atypical", 
        "3": "non-related", 
        "4": "non-symptomatic"},              
    "resting_analysis_results": {
        "0": "normal", 
        "1": "abnormal", 
        "2": "critical"},
    "chance_of_breakdown": {
        "0": "low", # less -> low
        "1": "high",} # more -> high
}

temp_df_count = df_count

for column, mapping in category_map.items():
    if column in df.columns:
        df = df.replace(to_replace=mapping, subset=[column])
        
        valid_values = list(mapping.values())

        df = df.filter(df[column].isin(valid_values))

df_count = df.count()
        
print(f"Number of dropped rows: {temp_df_count - df_count}")
print("Remaining rows:")
display(df)

In [0]:
# filter out records with more than 120k operating hours

temp_df_count = df_count
df = df.filter(df["operating_hours"] <= 120000)
df_count = df.count()

print(f"Number of dropped rows: {temp_df_count - df_count}")
print("Remaining rows:")
display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("gold_data")

### (N)MI Matrix
Creating a table containing (Normalized) Mutual Information scores between features for dashboard.
**This pandas based approach is suitable for only smaller datasets, for larger ones, distributed processing would be more efficient.**


In [0]:
df = spark.table("gold_data")

df = df.toPandas()

# [feature column name, is discrete, short name for dashboard ] 
feature_info = [
  ["operating_hours", False, "Op. Hours"],
  ["piston_material", True, "Pist. Mat."],
  ["combustion_issue_type", True, "Comb. Iss. Type"],
  ["break_mean_effective_pressure", False, "BMEP"],
  ["natural_gas_impurities_nmol", False, "Nat. Gas. Imp."],
  ["had_past_damages", True, "Past. Dmg."],
  ["resting_analysis_results", True, "Rest. An. Res."],
  ["maximum_rotations_per_minute", False, "Max. RPM."],
  ["had_full_load_issues", True, "Full Load Iss."],
  ["number_of_unplanned_events", False, "Unpl. Ev. No."],
  ["number_of_turbo_chargers", True, "Tur. Charg. No."],
  ["chance_of_breakdown", True, "Bd. Chance"]
]

discrete_features = [item[0] for item in feature_info if item[1] == True]
discrete_features = df[discrete_features].select_dtypes(exclude=["number"]).columns.tolist()
encoder = OrdinalEncoder()
df[discrete_features] = encoder.fit_transform(df[discrete_features])

mi_matrix = []

seed = 42
for x_feature, x_discrete, x_name in feature_info:
  for y_feature, y_discrete, y_name in feature_info:
    if y_discrete:
      mi = mutual_info_classif(
        df[[x_feature]], 
        df[y_feature], 
        discrete_features = [x_discrete],
        random_state = seed
      )
    else:
      mi = mutual_info_regression(
        df[[x_feature]], 
        df[y_feature], 
        discrete_features = [x_discrete],
        random_state = seed
      )
    mi_matrix.append({
        'x': x_name,
        'y': y_name,
        'mi': mi[0]
    })

mi_matrix_df = pd.DataFrame(mi_matrix)

entropies = mi_matrix_df[mi_matrix_df['x'] == mi_matrix_df['y']].set_index('x')['mi'].to_dict()

def get_nmi(row):
    h_x = entropies.get(row['x'], 0)
    h_y = entropies.get(row['y'], 0)
    
    if h_x == 0 or h_y == 0:
        return 0.0
    
    nmi = row['mi'] / np.sqrt(h_x * h_y)
    
    return nmi
  
mi_matrix_df['nmi'] = mi_matrix_df.apply(get_nmi, axis=1)
mi_matrix_df = mi_matrix_df.sort_values(by='nmi', ascending=False).reset_index(drop=True)

display(mi_matrix_df)

mi_matrix_df = spark.createDataFrame(mi_matrix_df)

mi_matrix_df.write.format("delta").mode("overwrite").saveAsTable("mi_matrix")
  